In [5]:
import torch
import torch.nn.functional as F
import cv2
import numpy as np
import os
from torchvision import models, transforms
from PIL import Image
from tqdm import tqdm

# ===================== CONFIG =====================
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

# ── Two separate test roots ──────────────────────────────────────────────────
# Digital (RGB) images live here:
DIGITAL_TEST_DIR  = r"/Users/dhruviramaiya/Downloads/Mtech Major Project/Dataset/Maturity/train test val split for digital/test"
# Thermal images live in the SEPARATE thermal split folder:
THERMAL_TEST_DIR  = r"/Users/dhruviramaiya/Downloads/Mtech Major Project/Dataset/Maturity/train test val split  thermal/test"

OUTPUT_DIR = "/Users/dhruviramaiya/Downloads/Mtech Major Project/XAI_Multimodal_2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_SIZE = 224
classes = ["mature", "semi_mature", "immature"]

# ===================== TRANSFORM =====================
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ===================== LOAD MODELS =====================

# RGB MODEL (ResNet50)
# Checkpoint was saved with fc as Sequential → keys are fc.1.weight / fc.1.bias
rgb_model = models.resnet50(pretrained=False)
in_features = rgb_model.fc.in_features
rgb_model.fc = torch.nn.Sequential(
    torch.nn.Identity(),                   # index 0
    torch.nn.Linear(in_features, 3)        # index 1  ← matches fc.1.weight/bias
)
rgb_model.load_state_dict(
    torch.load(
        r"/Users/dhruviramaiya/Downloads/Mtech Major Project/DeepLearningModels/digital_best_resnet50_guava.pth",
        map_location=DEVICE,
        weights_only=False
    )
)
rgb_model.to(DEVICE).eval()

# THERMAL MODEL (DenseNet121)
thermal_model = models.densenet121(pretrained=False)
thermal_model.classifier = torch.nn.Linear(thermal_model.classifier.in_features, 3)
thermal_model.load_state_dict(
    torch.load(
        "/Users/dhruviramaiya/Downloads/Mtech Major Project/DeepLearningModels/densenet121_thermal_best.pth",
        map_location=DEVICE,
        weights_only=False
    )
)
thermal_model.to(DEVICE).eval()

# ===================== HOOK STORAGE =====================
rgb_gradients,     rgb_activations     = [], []
thermal_gradients, thermal_activations = [], []

# ===================== HOOK FUNCTIONS =====================
def rgb_forward_hook(module, input, output):
    rgb_activations.append(output)

def rgb_backward_hook(module, grad_in, grad_out):
    rgb_gradients.append(grad_out[0])

def thermal_forward_hook(module, input, output):
    thermal_activations.append(output)

def thermal_backward_hook(module, grad_in, grad_out):
    thermal_gradients.append(grad_out[0])

# Attach hooks
rgb_model.layer4[-1].register_forward_hook(rgb_forward_hook)
rgb_model.layer4[-1].register_backward_hook(rgb_backward_hook)

thermal_model.features[-1].register_forward_hook(thermal_forward_hook)
thermal_model.features[-1].register_backward_hook(thermal_backward_hook)

# ===================== GRAD-CAM FUNCTION =====================
def generate_cam(model, gradients, activations, input_tensor, class_idx):
    gradients.clear()
    activations.clear()

    output = model(input_tensor)
    model.zero_grad()
    output[0, class_idx].backward()

    grad = gradients[0].cpu().data.numpy()[0]   # (C, H, W)
    act  = activations[0].cpu().data.numpy()[0]  # (C, H, W)

    weights = np.mean(grad, axis=(1, 2))          # (C,)

    cam = np.zeros(act.shape[1:], dtype=np.float32)
    for i, w in enumerate(weights):
        cam += w * act[i]

    cam = np.maximum(cam, 0)
    cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
    cam -= cam.min()
    cam /= (cam.max() + 1e-8)

    return cam

# ===================== OVERLAY =====================
def overlay(img, cam):
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap  = heatmap / 255.0
    img_norm = img     / 255.0
    combined = heatmap + img_norm
    combined /= combined.max()
    return np.uint8(255 * combined)

# ===================== TEST LOOP =====================
for cls in classes:

    # ── Paths for this class ─────────────────────────────────────────────────
    rgb_dir     = os.path.join(DIGITAL_TEST_DIR,  cls)   # digital test/<cls>
    thermal_dir = os.path.join(THERMAL_TEST_DIR,  cls)   # thermal test/<cls>

    if not os.path.isdir(rgb_dir):
        print(f"⚠️  RGB folder not found, skipping: {rgb_dir}")
        continue
    if not os.path.isdir(thermal_dir):
        print(f"⚠️  Thermal folder not found, skipping: {thermal_dir}")
        continue

    save_cls = os.path.join(OUTPUT_DIR, cls)
    os.makedirs(save_cls, exist_ok=True)

    img_names = [
        f for f in os.listdir(rgb_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))
    ]

    for img_name in tqdm(img_names, desc=f"{cls}"):

        rgb_img_path     = os.path.join(rgb_dir,     img_name)
        thermal_img_path = os.path.join(thermal_dir, img_name)

        # Check matching thermal image exists
        if not os.path.exists(thermal_img_path):
            print(f"⚠️  No matching thermal image for {img_name}, skipping.")
            continue

        # ── Load images ───────────────────────────────────────────────────────
        rgb_img     = Image.open(rgb_img_path).convert("RGB")
        thermal_img = Image.open(thermal_img_path).convert("RGB")

        rgb_orig     = cv2.resize(cv2.imread(rgb_img_path),     (IMG_SIZE, IMG_SIZE))
        thermal_orig = cv2.resize(cv2.imread(thermal_img_path), (IMG_SIZE, IMG_SIZE))

        rgb_tensor     = transform(rgb_img).unsqueeze(0).to(DEVICE)
        thermal_tensor = transform(thermal_img).unsqueeze(0).to(DEVICE)

        # ── Fused prediction (no grad needed here) ────────────────────────────
        with torch.no_grad():
            rgb_out     = rgb_model(rgb_tensor)
            thermal_out = thermal_model(thermal_tensor)

        fused_out  = (rgb_out + thermal_out) / 2
        pred_class = torch.argmax(fused_out, dim=1).item()

        # ── Grad-CAM (fresh tensors with grad enabled) ─────────────────────────
        rgb_tensor_grad     = rgb_tensor.clone().detach().requires_grad_(True)
        thermal_tensor_grad = thermal_tensor.clone().detach().requires_grad_(True)

        rgb_cam     = generate_cam(rgb_model,     rgb_gradients,     rgb_activations,
                                   rgb_tensor_grad,     pred_class)
        thermal_cam = generate_cam(thermal_model, thermal_gradients, thermal_activations,
                                   thermal_tensor_grad, pred_class)

        # ── Overlay + save ─────────────────────────────────────────────────────
        rgb_result     = overlay(rgb_orig,     rgb_cam)
        thermal_result = overlay(thermal_orig, thermal_cam)

        combined = np.hstack((rgb_result, thermal_result))
        cv2.imwrite(os.path.join(save_cls, img_name), combined)

print("✅ Multimodal Grad-CAM completed!")

mature:   2%|▏         | 11/485 [00:01<01:02,  7.63it/s]

⚠️  No matching thermal image for mature_2905.jpg, skipping.


mature:   4%|▍         | 20/485 [00:02<00:46,  9.94it/s]

⚠️  No matching thermal image for mature_3165.jpg, skipping.


mature:   6%|▌         | 29/485 [00:03<00:46,  9.75it/s]

...

⚠️  No matching thermal image for immature_2883.jpg, skipping.


immature:  99%|█████████▉| 485/490 [01:00<00:00,  9.77it/s]

⚠️  No matching thermal image for immature_3022.jpg, skipping.


immature: 100%|██████████| 490/490 [01:00<00:00,  8.07it/s]

✅ Multimodal Grad-CAM completed!
